In [ ]:
# Install Dependencies
!pip install -q transformers==4.47.0
!pip install -q peft==0.14.0
!pip install -q trl==0.13.0
!pip install -q bitsandbytes==0.45.0
!pip install -q datasets==3.2.0
!pip install -q accelerate==1.2.0

In [ ]:
# Cell 2 — All Imports
import json
import random
import pandas as pd
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer
import torch

In [ ]:
# Cell 3 — Constants & Config

# Model
BASE_MODEL      = "meta-llama/Llama-3.2-1B-Instruct"
ADAPTER_OUT_DIR = "./qlora-sales-qualifier"

# Data
NUM_SAMPLES     = 500
TRAIN_SPLIT     = 0.85  # 425 train / 75 eval

# Training
LORA_R          = 8
LORA_ALPHA      = 16
LORA_DROPOUT    = 0.05
LEARNING_RATE   = 2e-4
EPOCHS          = 3
BATCH_SIZE      = 4
MAX_SEQ_LENGTH  = 512

# Reproducibility
SEED            = 42
random.seed(SEED)

In [ ]:
# Synthetic Data Generation

INDUSTRIES    = ["SaaS", "FinTech", "Healthcare", "Retail", "Manufacturing", "Logistics", "EdTech"]
COMPANY_SIZES = ["1-10", "11-50", "51-200", "201-500", "501-1000", "1000+"]
LEAD_SOURCES  = ["Website", "LinkedIn", "Referral", "Cold Outreach", "Trade Show", "Webinar"]
PAIN_POINTS   = [
    "manual reporting", "poor lead tracking", "slow onboarding",
    "high churn", "lack of visibility", "disconnected tools", "scaling issues"
]

def assign_label_and_score(lead: dict) -> tuple:
    score = 0

    if lead["budget_usd"] >= 50000:   score += 30
    elif lead["budget_usd"] >= 20000: score += 18
    else:                             score += 5

    if lead["company_size"] in ["201-500", "501-1000", "1000+"]: score += 20
    elif lead["company_size"] in ["51-200"]:                     score += 12
    else:                                                         score += 4

    score += min(lead["pages_visited"] * 2, 10)
    score += min(lead["emails_opened"] * 3, 15)
    if lead["requested_demo"]:   score += 15
    if lead["has_pain_point"]:   score += 10

    score = min(score, 100)

    if score >= 65:   label = "Hot"
    elif score >= 35: label = "Warm"
    else:             label = "Cold"

    return label, score


def generate_hot_lead() -> dict:
    return {
        "industry":       random.choice(INDUSTRIES),
        "company_size":   random.choice(["201-500", "501-1000", "1000+"]),
        "lead_source":    random.choice(LEAD_SOURCES),
        "budget_usd":     random.choice([50000, 75000, 100000]),
        "pages_visited":  random.randint(6, 10),
        "emails_opened":  random.randint(3, 5),
        "requested_demo": True,
        "has_pain_point": True,
        "pain_point":     random.choice(PAIN_POINTS),
    }

def generate_warm_lead() -> dict:
    return {
        "industry":       random.choice(INDUSTRIES),
        "company_size":   random.choice(["51-200", "201-500"]),
        "lead_source":    random.choice(LEAD_SOURCES),
        "budget_usd":     random.choice([20000, 35000]),
        "pages_visited":  random.randint(3, 6),
        "emails_opened":  random.randint(1, 3),
        "requested_demo": random.choice([True, False]),
        "has_pain_point": random.choice([True, False]),
        "pain_point":     random.choice(PAIN_POINTS),
    }

def generate_cold_lead() -> dict:
    return {
        "industry":       random.choice(INDUSTRIES),
        "company_size":   random.choice(["1-10", "11-50"]),
        "lead_source":    random.choice(LEAD_SOURCES),
        "budget_usd":     random.choice([5000, 10000]),
        "pages_visited":  random.randint(1, 3),
        "emails_opened":  random.randint(0, 1),
        "requested_demo": False,
        "has_pain_point": False,
        "pain_point":     random.choice(PAIN_POINTS),
    }


# Generate balanced dataset (250 per label)
per_class = NUM_SAMPLES // 3
leads = []
for fn in [generate_hot_lead, generate_warm_lead, generate_cold_lead]:
    batch = [fn() for _ in range(per_class)]
    for lead in batch:
        lead["label"], lead["score"] = assign_label_and_score(lead)
    leads.extend(batch)

random.shuffle(leads)
df = pd.DataFrame(leads)

print(df["label"].value_counts())
print(df.head(3))

In [ ]:
#  Prompt Formatting (instruction-tuning pairs)

SYSTEM_PROMPT = (
    "You are a sales qualification assistant. "
    "Given CRM data about an inbound lead, return a qualification label (Hot, Warm, or Cold) "
    "and a score from 0 to 100. Respond in this exact format:\n"
    "Label: <Hot|Warm|Cold>\nScore: <0-100>\nReason: <one sentence>"
)

def format_prompt(lead: dict) -> str:
    user_msg = (
        f"Industry: {lead['industry']}\n"
        f"Company Size: {lead['company_size']} employees\n"
        f"Lead Source: {lead['lead_source']}\n"
        f"Budget: ${lead['budget_usd']:,}\n"
        f"Pages Visited: {lead['pages_visited']}\n"
        f"Emails Opened: {lead['emails_opened']}\n"
        f"Requested Demo: {lead['requested_demo']}\n"
        f"Has Pain Point: {lead['has_pain_point']} ({lead['pain_point']})"
    )
    assistant_msg = (
        f"Label: {lead['label']}\n"
        f"Score: {lead['score']}\n"
        f"Reason: This lead scores {lead['score']}/100 based on budget, engagement, and fit signals."
    )
    return (
        f"<|system|>\n{SYSTEM_PROMPT}\n"
        f"<|user|>\n{user_msg}\n"
        f"<|assistant|>\n{assistant_msg}"
    )


df["text"] = df.apply(format_prompt, axis=1)

# Train / eval split
split_idx = int(len(df) * TRAIN_SPLIT)
train_dataset = Dataset.from_pandas(df[["text"]].iloc[:split_idx].reset_index(drop=True))
eval_dataset  = Dataset.from_pandas(df[["text"]].iloc[split_idx:].reset_index(drop=True))

print(f"Train samples : {len(train_dataset)}")
print(f"Eval samples  : {len(eval_dataset)}")
print("\nSample prompt:\n")
print(df["text"].iloc[0])

In [ ]:
from huggingface_hub import login; login()

In [ ]:
# Load Base Model + Tokenizer in 4-bit



bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
)
model.config.use_cache = False

print(f"Model loaded: {BASE_MODEL}")
print(f"Device map  : {model.hf_device_map}")

In [ ]:
#  LoRA Config

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


In [ ]:
#  Fine-tune with SFTTrainer

from trl import SFTConfig

training_args = SFTConfig(
    output_dir=ADAPTER_OUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=4,
    learning_rate=LEARNING_RATE,
    fp16=True,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    max_seq_length=MAX_SEQ_LENGTH,
    seed=SEED,
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
)

trainer.train()

In [ ]:
#  Save adapter and tokenizer

model.save_pretrained(ADAPTER_OUT_DIR)
tokenizer.save_pretrained(ADAPTER_OUT_DIR)

print(f"Adapter saved to: {ADAPTER_OUT_DIR}")
!ls {ADAPTER_OUT_DIR}

In [ ]:
# Inference & Evaluation

from peft import PeftModel

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    ),
    device_map="auto",
)
fine_tuned_model = PeftModel.from_pretrained(base_model, ADAPTER_OUT_DIR)
fine_tuned_model.eval()

def qualify_lead(lead: dict) -> str:
    prompt = (
        f"<|system|>\n{SYSTEM_PROMPT}\n"
        f"<|user|>\n"
        f"Industry: {lead['industry']}\n"
        f"Company Size: {lead['company_size']} employees\n"
        f"Lead Source: {lead['lead_source']}\n"
        f"Budget: ${lead['budget_usd']:,}\n"
        f"Pages Visited: {lead['pages_visited']}\n"
        f"Emails Opened: {lead['emails_opened']}\n"
        f"Requested Demo: {lead['requested_demo']}\n"
        f"Has Pain Point: {lead['has_pain_point']} ({lead['pain_point']})\n"
        f"<|assistant|>\n"
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(fine_tuned_model.device)
    with torch.no_grad():
        outputs = fine_tuned_model.generate(
            **inputs,
            max_new_tokens=80,
            do_sample=False,
            repetition_penalty=1.3,
            eos_token_id=tokenizer.eos_token_id,
        )
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Extract only the assistant response and trim after "Reason" sentence
    assistant_response = response.split("<|assistant|>")[-1].strip()
    lines = [l for l in assistant_response.splitlines() if l.startswith(("Label:", "Score:", "Reason:"))]
    return "\n".join(lines[:3])


# Test leads
test_leads = [
    {"industry": "SaaS", "company_size": "201-500", "lead_source": "Referral",
     "budget_usd": 75000, "pages_visited": 8, "emails_opened": 4,
     "requested_demo": True, "has_pain_point": True, "pain_point": "high churn"},

    {"industry": "Retail", "company_size": "1-10", "lead_source": "Cold Outreach",
     "budget_usd": 5000, "pages_visited": 2, "emails_opened": 0,
     "requested_demo": False, "has_pain_point": False, "pain_point": "manual reporting"},

    {"industry": "FinTech", "company_size": "51-200", "lead_source": "Webinar",
     "budget_usd": 20000, "pages_visited": 5, "emails_opened": 2,
     "requested_demo": False, "has_pain_point": True, "pain_point": "disconnected tools"},
]

for i, lead in enumerate(test_leads, 1):
    print(f"--- Lead {i} ---")
    print(qualify_lead(lead))
    print()